## This file creates Hospital Episode Statistics synthetic data  

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import os

In [3]:
os.chdir(r"C:\Users\uctvjla\OneDrive - University College London\Documents\GitHub\ECHILD_Synthetic\Python")

In [4]:

def random_epikey_generator(n):
    epikeys = random.sample(range(100000000000, 200000000000), n)
    return epikeys

if not (os.path.exists("2_spine_jupyter.csv") and os.path.exists("1_people_jupyter.csv")):
    source("2_spine.ipynb")
else:
    people = pd.read_csv("1_people_jupyter.csv")
    people['gestat'] = people['gestational_age']
    
    people['diag_01'] = "Z38"
    people['startage'] = 7001
    people['epitype'] = 3
    people['admimeth'] = 82
    people['epiorder'] = 1
    
    people['ethnos'] = people['ethnicity']
    ethnos_mapping = {
        'white_british': 'A',
        'white_irish': 'B',
        'white_other': 'C',
        'mixed_white_black_carribean': 'D',
        'mixed_white_black_african': 'E',
        'mixed_white_asian': 'F',
        'mixed_other': 'G',
        'asian_indian': 'H',
        'asian_pakistani': 'J',
        'asian_bangleshi': 'K',
        'asian_other': 'L',
        'black_carribean': 'M',
        'black_african': 'N',
        'black_other': 'P',
        'asian_chinese': 'R',
        'other_any_other': 'S',
        'other_arab': 'S'
    }
    people['ethnos'] = people['ethnos'].map(ethnos_mapping)
    
    people['epikey'] = random_epikey_generator(len(people))
    
    people['year_of_birth'] = np.where(people['month_of_birth'].isin(['sep', 'oct', 'nov', 'dec']), 2004, 2005)
    
    people['dob'] = "15" + people['month_of_birth'] + people['year_of_birth'].astype(str)
    people['dob_reformatted'] = pd.to_datetime(people['dob'], format="%d%b%Y")
    people['epistart'] = people['dob_reformatted']
    people['admistart'] = people['dob_reformatted']
    
    people['sex'] = people['gender'] + 1
    
    people['gestational_age'] = pd.Categorical(people['gestational_age'],
                                               categories=['less_than_28', '28_29', '30_31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42'],
                                               ordered=True)
    
    people['gestational_age_num'] = people['gestational_age'].cat.codes + 1
    people['gestational_age_num_reverse'] = max(people['gestational_age_num']) - people['gestational_age_num']
    people['gestational_age_num_reverse_half'] = np.round(people['gestational_age_num_reverse'] / 2)
    people['gestational_age_random'] = np.random.choice([0, 1], replace=True, size=len(people))
    people['disdate'] = people['admistart'] + timedelta(days=1) + pd.to_timedelta(people['gestational_age_num_reverse_half'] * people['gestational_age_random'], unit='d')

    people['epiend'] = people['disdate']
    
    people['diff'] = (people['disdate'] - people['admistart']).dt.days
    people['diag_02'] = np.where(people['diff'] >= 3, np.random.choice([np.nan, "Q43", "Q24", "Q35", "Q37", "A75", "P961", "Z590"], replace=True, size=len(people)), np.nan)
    
    imd_deciles_mapping = {
        1: "Most Deprived 10%",
        2: "More Deprived 10% - 20%",
        3: "More Deprived 20% - 30%",
        4: "More Deprived 30% - 40%",
        5: "More Deprived 40% - 50%",
        6: "Less Deprived 40% - 50%",
        7: "Less Deprived 30% - 40%",
        8: "Less Deprived 20% - 30%",
        9: "Less Deprived 10% - 20%",
        10: "Least Deprived 10%"
    }
    people['imd04_decile'] = people['imd_deciles'].map(imd_deciles_mapping)
    
    resgor_mapping = {
        'north_east': 'A',
        'north_west': 'B',
        'yorkshire_humber': 'D',
        'east_midlands': 'E',
        'west_midlands': 'F',
        'east': 'G',
        'london': 'H',
        'south_east': 'J',
        'south_west': 'K',
        'scotland': 'S',
        'wales': 'W'
    }
    people['resgor'] = people['resgor_birth'].map(resgor_mapping)
    
    hes = people[['tokenid', 'epikey', 'epitype', 'epiorder',
                  'epistart', 'epiend', 'admistart', 'disdate',
                  'admimeth', 'sex', 'startage','ethnos', 'resgor', 'imd04_decile',
                  'gestat', 'diag_01', 'diag_02']]
    
    hes.to_csv("4_HES_jupyter.csv", index=False)
